In [ ]:
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score, recall_score, precision_score, confusion_matrix
)
import matplotlib.pyplot as plt
from collections import defaultdict
import joblib
from scipy.stats import skew, kurtosis, shapiro
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('demographics_data.csv')
df = df.drop(columns=['Unnamed: 0','SEQN'])
# Create depression category
df['Depression_Category'] = df["Total_Depression_Score"].apply(lambda x: 1 if x >= 10 else 0)

In [ ]:
# Create new features
df['circadian_ratio'] = (df['morning'] + df['afternoon']) / (df['evening'] + df['night'])
df['composite_mims'] = df['AVG_PEAK30_MIMS'] / df['DAILY_MEAN_MIMS']
df['high_v_low'] = df['Moderate'] / (df['Light'] + df['Sedentary'])
df['Income_per_person'] = df['HH_INCOME'] / df['HH_NUMBER']

In [ ]:
both_features = ['SEX', 'AGE', 'RACE', 'CITIZENSHIP', 'EDUCATION', 'MARITAL_STATUS','HH_NUMBER', 'HH_INCOME', 'RATIO_POVERTY','Light', 'Moderate', 'Sedentary', 'DAILY_MEAN_MIMS', 'afternoon','evening', 'morning', 'night', 'AVG_PEAK30_MIMS', 'AVG_SLEEP_HOURS','SLEEP_SD_HOURS', 'Avg_wake_bouts','circadian_ratio','composite_mims','high_v_low', 'Income_per_person', 'Depression_Category']

In [ ]:
both_df_features = df[both_features]

In [ ]:
# TARGET
y_both = both_df_features["Depression_Category"]

# ALL predictors
X_both = both_df_features.drop(columns=["Depression_Category"])

normal_vars_both = []
minmax_vars_both = []
robust_vars_both = []

for feat in X_both.columns:
    x = both_df_features[feat].dropna().values

    # 1. Shapiro normality test
    p_norm = shapiro(x)[1]

    # 2. Skewness
    sk = skew(x)

    # 3. Outlier ratio using IQR
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_ratio = np.mean((x < lower) | (x > upper))

    # --- Decision rules ---
    if p_norm > 0.05:  
        # approximately normal
        normal_vars_both.append(feat)

    elif abs(sk) >= 2 or outlier_ratio > 0.05:
        # heavy skew or many outliers → use RobustScaler
        robust_vars_both.append(feat)

    else:
        # non-normal but not extreme → use MinMaxScaler
        minmax_vars_both.append(feat)

print("StandardScaler vars:", normal_vars_both)
print("MinMaxScaler vars:", minmax_vars_both)
print("RobustScaler vars:", robust_vars_both)

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("zscore", StandardScaler(), normal_vars_both),
        ("minmax", MinMaxScaler(), minmax_vars_both),
        ("robust", RobustScaler(), robust_vars_both)
    ],
    remainder="passthrough"
)

In [ ]:
# 80% train, 20% temp
X_train, X_test, y_train, y_test = train_test_split(
    X_both, y_both, test_size=0.10, stratify=y_both, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
# Bin AGE and RATIO_POVERTY before grouping
X_test["AGE"] = pd.cut(
    X_test["AGE"],
    bins=[20, 30, 40, 50, 60, 70, 120],
    labels=[20, 30, 40, 50, 60, 70],
    include_lowest=True
)

X_test["RATIO_POVERTY"] = pd.cut(X_test["RATIO_POVERTY"], bins=[0,1,2,3,4,5],
                           labels=[0,1,2,3,4])

In [ ]:
best_model = joblib.load("accel_best_model.pkl")
best_model_demo = joblib.load("demo_best_model.pkl")
best_model_both = joblib.load("combined_best_model.pkl")
best_model_smote = joblib.load("accel_best_model_sampling.pkl")
best_model_demo_smote = joblib.load("demo_best_model_sampling.pkl")
best_model_both_smote = joblib.load("combined_best_model_sampling.pkl")

In [ ]:
models = {
    "accel_only": best_model['model'],
    "demo_only": best_model_demo['model'],
    "combined": best_model_both['model'],
    "accel_only_smote": best_model_smote['model'],
    "demo_only_smote": best_model_demo_smote['model'],
    "combined_undersample": best_model_both_smote['model']
}

group_cols = ['SEX', 'AGE', 'RACE', 'CITIZENSHIP', 'EDUCATION', 'MARITAL_STATUS','HH_NUMBER', 'HH_INCOME', 'RATIO_POVERTY']   # modify as needed
fairness_outputs = {}
shap_outputs = {}

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score
import pandas as pd

def stratified_f1_all(models, X_test, y_test, group_cols):
    """
    Computes and prints F1 scores for all models and groups,
    then plots a bar plot for each group.
    """
    all_data = []

    # Compute F1 scores for all models and groups
    for group_col in group_cols:
        for model_name, model in models.items():
            df_temp = X_test[[group_col]].copy()
            df_temp["y_true"] = y_test.values
            df_temp["y_pred"] = model.predict(X_test)

            subgroups = sorted(df_temp[group_col].dropna().unique())

            for subgroup in subgroups:
                subset = df_temp[df_temp[group_col] == subgroup]
                if subset["y_true"].nunique() < 2:
                    f1 = 0
                else:
                    f1 = f1_score(subset["y_true"], subset["y_pred"], zero_division=0)
                all_data.append([group_col, subgroup, model_name, f1])

    # Convert to DataFrame
    f1_df = pd.DataFrame(all_data, columns=["Group", "Subgroup", "Model", "F1"])

    # Print all values
    print("\n=== F1 Scores for All Groups and Models ===")
    for group_col in group_cols:
        print(f"\n--- {group_col} ---")
        group_df = f1_df[f1_df["Group"] == group_col]
        for model_name in group_df["Model"].unique():
            model_scores = group_df[group_df["Model"] == model_name][["Subgroup","F1"]]
            print(f"\n{model_name}:")
            print(model_scores.set_index("Subgroup"))

    # Plot all groups
    for group_col in group_cols:
        plt.figure(figsize=(8,5))
        group_df = f1_df[f1_df["Group"] == group_col]
        sns.barplot(x="Subgroup", y="F1", hue="Model", data=group_df, palette="colorblind")
        plt.title(f"F1 Score by {group_col}")
        plt.ylim(0,1)
        plt.ylabel("F1 Score")
        plt.xlabel(group_col)
        plt.legend(title="Model")
        plt.show()


# Run for all groups
stratified_f1_all(models, X_test, y_test, group_cols)

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def plot_feature_importance_permutation(model, X_test, y_test, feature_names, model_name, top_n=10, n_repeats=10, random_state=42):
    """
    Plots feature importance using permutation importance, works for any model.
    """
    result = permutation_importance(model, X_test, y_test, n_repeats=n_repeats, random_state=random_state, scoring='f1')
    
    importances = result.importances_mean
    fi_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False).head(top_n)
    
    plt.figure(figsize=(8,6))
    sns.barplot(x="importance", y="feature", data=fi_df, palette="magma")
    plt.title(f"Top {top_n} Feature Importance (Permutation) — {model_name}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Demographics only
plot_feature_importance_permutation(models["demo_only"], 
                                    X_test[['SEX','AGE','RACE','CITIZENSHIP','EDUCATION','MARITAL_STATUS','HH_NUMBER','HH_INCOME','RATIO_POVERTY','Income_per_person']], 
                                    y_test,
                                    ['SEX','AGE','RACE','CITIZENSHIP','EDUCATION','MARITAL_STATUS','HH_NUMBER','HH_INCOME','RATIO_POVERTY','Income_per_person'],
                                    "Demographics Only")

# Activity only
plot_feature_importance_permutation(models["accel_only"],
                                    X_test[['Light','Moderate','Sedentary','DAILY_MEAN_MIMS','afternoon','evening','morning','night',
                                            'AVG_PEAK30_MIMS','AVG_SLEEP_HOURS','SLEEP_SD_HOURS','Avg_wake_bouts','circadian_ratio','composite_mims','high_v_low']],
                                    y_test,
                                    ['Light','Moderate','Sedentary','DAILY_MEAN_MIMS','afternoon','evening','morning','night',
                                     'AVG_PEAK30_MIMS','AVG_SLEEP_HOURS','SLEEP_SD_HOURS','Avg_wake_bouts','circadian_ratio','composite_mims','high_v_low'],
                                    "Activity Only")

# Combined
plot_feature_importance_permutation(models["combined"], 
                                    X_test[['SEX','AGE','RACE','CITIZENSHIP','EDUCATION','MARITAL_STATUS','HH_NUMBER','HH_INCOME','RATIO_POVERTY','Income_per_person','Light','Moderate','Sedentary','DAILY_MEAN_MIMS','afternoon','evening','morning','night',
                                            'AVG_PEAK30_MIMS','AVG_SLEEP_HOURS','SLEEP_SD_HOURS','Avg_wake_bouts','circadian_ratio','composite_mims','high_v_low']], 
                                    y_test,
                                    ['SEX','AGE','RACE','CITIZENSHIP','EDUCATION','MARITAL_STATUS','HH_NUMBER','HH_INCOME','RATIO_POVERTY','Income_per_person','Light','Moderate','Sedentary','DAILY_MEAN_MIMS','afternoon','evening','morning','night',
                                            'AVG_PEAK30_MIMS','AVG_SLEEP_HOURS','SLEEP_SD_HOURS','Avg_wake_bouts','circadian_ratio','composite_mims','high_v_low'],
                                    "Combined")
# Demographics only
plot_feature_importance_permutation(models["demo_only_smote"], 
                                    X_test[['SEX','AGE','RACE','CITIZENSHIP','EDUCATION','MARITAL_STATUS','HH_NUMBER','HH_INCOME','RATIO_POVERTY','Income_per_person']], 
                                    y_test,
                                    ['SEX','AGE','RACE','CITIZENSHIP','EDUCATION','MARITAL_STATUS','HH_NUMBER','HH_INCOME','RATIO_POVERTY','Income_per_person'],
                                    "Demographics Only")

# Activity only
plot_feature_importance_permutation(models["accel_only_smote"],
                                    X_test[['Light','Moderate','Sedentary','DAILY_MEAN_MIMS','afternoon','evening','morning','night',
                                            'AVG_PEAK30_MIMS','AVG_SLEEP_HOURS','SLEEP_SD_HOURS','Avg_wake_bouts','circadian_ratio','composite_mims','high_v_low']],
                                    y_test,
                                    ['Light','Moderate','Sedentary','DAILY_MEAN_MIMS','afternoon','evening','morning','night',
                                     'AVG_PEAK30_MIMS','AVG_SLEEP_HOURS','SLEEP_SD_HOURS','Avg_wake_bouts','circadian_ratio','composite_mims','high_v_low'],
                                    "Activity Only")

# Combined
plot_feature_importance_permutation(models["combined_undersample"], 
                                    X_test[['SEX','AGE','RACE','CITIZENSHIP','EDUCATION','MARITAL_STATUS','HH_NUMBER','HH_INCOME','RATIO_POVERTY','Income_per_person','Light','Moderate','Sedentary','DAILY_MEAN_MIMS','afternoon','evening','morning','night',
                                            'AVG_PEAK30_MIMS','AVG_SLEEP_HOURS','SLEEP_SD_HOURS','Avg_wake_bouts','circadian_ratio','composite_mims','high_v_low']], 
                                    y_test,
                                    ['SEX','AGE','RACE','CITIZENSHIP','EDUCATION','MARITAL_STATUS','HH_NUMBER','HH_INCOME','RATIO_POVERTY','Income_per_person','Light','Moderate','Sedentary','DAILY_MEAN_MIMS','afternoon','evening','morning','night',
                                            'AVG_PEAK30_MIMS','AVG_SLEEP_HOURS','SLEEP_SD_HOURS','Avg_wake_bouts','circadian_ratio','composite_mims','high_v_low'],
                                    "Combined")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

def get_permutation_importance_df(model, X, y, feature_names, n_repeats=10, random_state=42, scoring='f1'):
    """
    Returns a DataFrame of permutation importances for a model.
    """
    result = permutation_importance(model, X, y, n_repeats=n_repeats, random_state=random_state, scoring=scoring)
    return pd.DataFrame({
        'feature': feature_names,
        'importance': result.importances_mean
    }).set_index('feature')

# Define feature groups
demo_features = ['SEX','AGE','RACE','MARITAL_STATUS','Income_per_person','RATIO_POVERTY']
activity_features = ['SLEEP_SD_HOURS','Avg_wake_bouts','circadian_ratio','composite_mims','high_v_low']
combined_features = demo_features + activity_features

# Collect all models
model_dict = {
    'Demo Only': (models['demo_only'], demo_features),
    'Demo Only SMOTE': (models['demo_only_smote'], demo_features),
    'Activity Only': (models['accel_only'], activity_features),
    'Activity Only Undersample': (models['accel_only_smote'], activity_features),
    'Combined': (models['combined'], combined_features),
    'Combined Undersample': (models['combined_undersample'], combined_features)
}

# All features we want on the y-axis
all_features = combined_features  # demo + activity

# Compute permutation importances for all models
importance_df = pd.DataFrame(index=all_features)

for name, (model, features) in model_dict.items():
    df = get_permutation_importance_df(model, X_test[features], y_test, features)
    # Reindex to include all features, fill missing with 0
    df = df.reindex(all_features).fillna(0)
    importance_df[name] = df['importance']

# Plot heatmap with bigger labels
plt.figure(figsize=(8,10))
sns.heatmap(importance_df, annot=False, cmap="magma", fmt=".2f")

# Change x and y tick label names
x_labels = ['Demographics Only', 'Demographics Only (SMOTE)', 'Accelerometer Only', 'Accelerometer Only (Undersample)', 'Combined', 'Combined (Undersample)']
y_labels = ['Sex', 'Age', 'Race', 'Marital Status', 'Income per Person', 'Poverty Ratio',
            'Sleep Hours (SD)', 'Average Wake Bouts','Circadian Ratio','Composite MIMS','Activity Intensity Ratio']
plt.xticks(ticks=np.arange(len(importance_df.columns))+0.5, labels=x_labels, rotation=45, ha='right')
plt.yticks(ticks=np.arange(len(importance_df.index))+0.5, labels=y_labels, rotation=0)

# Increase title and axis label sizes
plt.xlabel("Model", fontsize=16)
plt.ylabel("Feature", fontsize=16)

# Increase tick label sizes
plt.xticks(fontsize=14, rotation=45)
plt.yticks(fontsize=14, rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Print the full permutation importance table
print(importance_df)

# Print features with permutation importance <= 0 for the combined models
for col in ["Combined", "Combined Undersample"]:
    if col not in importance_df.columns:
        print(f"\nColumn '{col}' not found in importance_df. Available: {list(importance_df.columns)}")
        continue
    nonpos = importance_df.loc[importance_df[col] <= 0, [col]].sort_values(col)
    print(f"\nFeatures with permutation importance <= 0 ({col}):")
    if nonpos.empty:
        print("  (none)")
    else:
        print(nonpos)

In [ ]:
from sklearn.metrics import confusion_matrix

# Initialize dictionary to store fairness metrics
fairness_outputs = {}

# Loop through each model
for model_name, model in models.items():
    fairness_outputs[model_name] = {}
    
    # Predict on your X_test
    y_pred = model.predict(X_test)
    
    # Loop through each group column
    for group_col in group_cols:
        fairness_outputs[model_name][group_col] = {}
        
        groups = X_test[group_col].unique()
        
        # Compute metrics for each group
        metrics = {}
        for g in groups:
            idx = X_test[group_col] == g
            y_true_group = y_test[idx]
            y_pred_group = y_pred[idx]
            
            # Avoid empty group
            if len(y_true_group) == 0:
                continue
            
            # Confusion matrix: TN, FP, FN, TP
            tn, fp, fn, tp = confusion_matrix(
                y_true_group, y_pred_group, labels=[0,1]
            ).ravel()
            
            # Sensitivity / TPR
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
            
            # False positive rate (FPR)
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
            
            # Positive Prediction Rate (PPR)
            ppr = (tp + fp) / len(y_true_group)
            
            metrics[g] = {
                "TPR": tpr,
                "FPR": fpr,
                "PPR": ppr
            }
        
        # If we have valid subgroup data:
        if metrics:
            tpr_values = [v["TPR"] for v in metrics.values()]
            fpr_values = [v["FPR"] for v in metrics.values()]
            ppr_values = [v["PPR"] for v in metrics.values()]

            # EOD (already implemented) = TPR max - TPR min
            fairness_outputs[model_name][group_col]["Equal_Opportunity_Diff"] = (
                max(tpr_values) - min(tpr_values)
            )
            
            # Demographic Parity Difference (PPD)
            fairness_outputs[model_name][group_col]["Demographic_Parity_Diff"] = (
                max(ppr_values) - min(ppr_values)
            )
            
            # --- NEW METRICS ---
            # 1. Average Odds Difference (AOD)
            # AOD = average of (TPR diff) and (FPR diff)
            aod = ((max(tpr_values) - min(tpr_values)) +
                   (max(fpr_values) - min(fpr_values))) / 2
            fairness_outputs[model_name][group_col]["Average_Odds_Diff"] = aod
            
            # 2. Statistical Parity Difference (SPD)
            # SPD = PPR_max - PPR_min
            spd = max(ppr_values) - min(ppr_values)
            fairness_outputs[model_name][group_col]["Statistical_Parity_Diff"] = spd
            
            # 3. Disparate Impact (DI)
            # DI = PPR_min / PPR_max
            di = min(ppr_values) / max(ppr_values) if max(ppr_values) > 0 else 0
            fairness_outputs[model_name][group_col]["Disparate_Impact"] = di

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Choose which metrics to plot (exclude Demographic_Parity_Diff)
metrics_to_plot = [
    "Equal_Opportunity_Diff",
    "Average_Odds_Diff",
    "Statistical_Parity_Diff",
    "Disparate_Impact"
]

for metric in metrics_to_plot:
    heatmap_data = {
        model_name: {
            group_col: fairness_outputs[model_name][group_col].get(metric, None)
            for group_col in fairness_outputs[model_name]
        }
        for model_name in fairness_outputs
    }

    df_metric = pd.DataFrame(heatmap_data).T

    plt.figure(figsize=(12,5))
    sns.heatmap(df_metric, annot=True, cmap="cividis",
                cbar_kws={'label': metric}, fmt=".2f", annot_kws={"size":12})
    
    # Customize x and y tick labels
    x_labels = ['Sex', 'Age', 'Race', 'Citizenship', 'Education', 'Marital Status', 'Household Number', 'Household Income', 'Poverty Ratio']
    plt.xticks(ticks=np.arange(len(df_metric.columns))+0.5, labels=x_labels, rotation=45, ha='right')
    y_labels = ['Accelerometer Only','Demographics Only', 'Combined', 'Accelerometer Only (SMOTE)', 'Demographics Only (SMOTE)', 'Combined (Undersample)']
    plt.yticks(ticks=np.arange(len(df_metric.index))+0.5, labels=y_labels, rotation=0)

    # Increase title and axis label sizes
    if metric == "Disparate_Impact":
        plt.title(f"{metric.replace('_', ' ')} by Model and Feature", fontsize=18)
    else:
        plt.title(f"{metric.replace('_', ' ')}erence by Model and Feature", fontsize=18)
    plt.ylabel("Model", fontsize=14)
    plt.xlabel("Feature", fontsize=14)

    # Change legend text
    if metric == "Disparate_Impact":
        plt.gca().collections[0].colorbar.set_label(f'{metric.replace("_", " ")}', fontsize=12)
    else:   
        plt.gca().collections[0].colorbar.set_label(f'{metric.replace("_", " ")}erence', fontsize=12)

    # Increase tick label sizes
    plt.xticks(fontsize=12, rotation=45)
    plt.yticks(fontsize=12, rotation=0)

    plt.tight_layout()
    plt.show()

In [ ]:
# ------------------------------------------------------------
# SHAP analysis (model behaviour explanation)
# Explains the underlying estimator inside the Pipeline after preprocessing.
# Produces a beeswarm summary + mean(|SHAP|) bar plot for class 1 (Depression=1).
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

# Map raw feature names to the same human-friendly labels used in the heatmap
_PRETTY_FEATURE = {
    # Demographics
    "SEX": "Sex",
    "AGE": "Age",
    "RACE": "Race",
    "CITIZENSHIP": "Citizenship",
    "EDUCATION": "Education",
    "MARITAL_STATUS": "Marital Status",
    "HH_NUMBER": "Household Number",
    "HH_INCOME": "Household Income",
    "RATIO_POVERTY": "Poverty Ratio",
    "Income_per_person": "Income per Person",
    # Activity / sleep
    "Light": "Light Activity",
    "Moderate": "Moderate Activity",
    "Sedentary": "Sedentary Activity",
    "DAILY_MEAN_MIMS": "Daily Mean MIMS",
    "afternoon": "Afternoon Activity",
    "evening": "Evening Activity",
    "morning": "Morning Activity",
    "night": "Night Activity",
    "AVG_PEAK30_MIMS": "Average Peak 30 MIMS",
    "AVG_SLEEP_HOURS": "Sleep Hours (Avg)",
    "SLEEP_SD_HOURS": "Sleep Hours (SD)",
    "Avg_wake_bouts": "Average Wake Bouts",
    "circadian_ratio": "Circadian Ratio",
    "composite_mims": "Composite MIMS",
    "high_v_low": "Activity Intensity Ratio",
}

def _pretty_feature_name(name: str) -> str:
    # Strip ColumnTransformer prefixes like `minmax__AGE` or `remainder__HH_INCOME`
    base = name.split("__", 1)[1] if "__" in name else name
    return _PRETTY_FEATURE.get(base, base)

def compute_shap(model, X, X_train=None, *, max_background=200, random_state=42):
    """Return (shap_values, X_transformed_df). Works for a Pipeline with steps
    ('preprocess', ...) and ('model', ...) as saved in your artifacts."""
    preprocess = None
    estimator = model
    if hasattr(model, "named_steps") and "model" in model.named_steps:
        preprocess = model.named_steps.get("preprocess")
        estimator = model.named_steps["model"]

    if preprocess is not None:
        X_trans = preprocess.transform(X)
        try:
            feature_names = list(preprocess.get_feature_names_out())
        except Exception:
            feature_names = [f"feat_{i}" for i in range(X_trans.shape[1])]
    else:
        X_trans = X
        feature_names = list(X.columns) if hasattr(X, "columns") else [f"feat_{i}" for i in range(X_trans.shape[1])]

    pretty_feature_names = [_pretty_feature_name(n) for n in feature_names]

    bg_trans = None
    if X_train is not None:
        if hasattr(X_train, "sample"):
            X_bg = X_train.sample(n=min(max_background, len(X_train)), random_state=random_state)
        else:
            X_bg = X_train[:max_background]
        bg_trans = preprocess.transform(X_bg) if preprocess is not None else X_bg

    # Prefer TreeExplainer for tree models; fall back to generic Explainer otherwise
    try:
        explainer = shap.TreeExplainer(estimator, data=bg_trans)
        shap_values = explainer.shap_values(X_trans, check_additivity=False)
    except Exception:
        masker = bg_trans if bg_trans is not None else X_trans
        explainer = shap.Explainer(estimator, masker)
        shap_values = explainer(X_trans)

    X_trans_df = pd.DataFrame(X_trans, columns=pretty_feature_names)
    return shap_values, X_trans_df

def _to_per_feature_shap(shap_obj):
    """Normalizes SHAP outputs to a 2D array (n_samples, n_features).
    Handles the case where TreeExplainer returns interaction tensors (n, p, p)."""
    if isinstance(shap_obj, np.ndarray) and shap_obj.ndim == 3:
        return shap_obj.sum(axis=2)
    return shap_obj

# Choose which model to explain
model_to_explain = "combined"  # if models is a dict, this key should exist
model = None

# This notebook later overwrites `models` (dict -> list) in a plotting cell, so be defensive.
if isinstance(globals().get("models", None), dict) and model_to_explain in models:
    model = models[model_to_explain]
elif isinstance(globals().get("model_dict", None), dict):
    # model_dict entries are (pipeline, feature_list) tuples from the permutation-importance section
    if "Combined" in model_dict and isinstance(model_dict["Combined"], tuple):
        model = model_dict["Combined"][0]
    elif "Combined" in model_dict:
        model = model_dict["Combined"]
if model is None:
    raise ValueError("Could not find a model pipeline to explain. Expected `models['combined']` or `model_dict['Combined']`.")

# Compute SHAP on the held-out test set (background sampled from training set)
shap_values, X_trans_df = compute_shap(model, X_test, X_train=X_train)

# SHAP plotting: for binary classifiers, focus on class 1 (positive class)
shap_to_plot = shap_values
if isinstance(shap_values, list) and len(shap_values) == 2:
    shap_to_plot = shap_values[1]
shap_to_plot = _to_per_feature_shap(shap_to_plot)

plt.figure()
shap.summary_plot(shap_to_plot, X_trans_df, show=False, max_display=20)
plt.title(f"SHAP Summary (beeswarm) — Combined")
plt.tight_layout()
plt.show()

plt.figure()
shap.summary_plot(shap_to_plot, X_trans_df, plot_type="bar", show=False, max_display=20)
plt.title(f"SHAP Feature Importance (mean |SHAP|) — Combined")
plt.tight_layout()
plt.show()

In [ ]:
# Redo the above heatmaps as a bar plot instead where the x is the feature and y is the metric value for each model.
# Choose which metrics to plot
metrics_to_plot = [
    "Equal_Opportunity_Diff",
    "Average_Odds_Diff",
    "Statistical_Parity_Diff",
    "Disparate_Impact"
]

# prefer a specific model order if available, otherwise use keys from fairness_outputs
preferred_order = [
    'Accelerometer Only','Demographics Only','Combined',
    'Accelerometer Only (Undersample)','Demographics Only (SMOTE)','Combined (Undersample)'
]
models = [m for m in preferred_order if m in fairness_outputs]
if not models:
    models = list(fairness_outputs.keys())

# get feature/order from the first model available
first_model = next(iter(fairness_outputs.values()))
features = list(first_model.keys())
feature_labels = ['Sex', 'Age', 'Race', 'Citizenship', 'Education', 'Marital Status', 'Household Number', 'Household Income', 'Poverty Ratio']


# create vertical subplots: one metric per row, shared x-axis and single legend
n_metrics = len(metrics_to_plot)
fig, axes = plt.subplots(n_metrics, 1, figsize=(16, 20), sharex=True)
if n_metrics == 1:
    axes = [axes]

# styling: colorblind-friendly palette
sns.set_style('whitegrid')
palette = sns.color_palette('colorblind', n_colors=len(models))

# generate subplot labels (A, B, C, ...)
labels = [chr(ord('A') + i) for i in range(n_metrics)]

for ax, metric, label in zip(axes, metrics_to_plot, labels):
    rows = []
    for model in models:
        for feat,feat_name in zip(features, feature_labels):
            val = fairness_outputs.get(model, {}).get(feat, {}).get(metric, np.nan)
            rows.append({'Feature': feat_name, 'Model': model, 'Value': val})
    df_plot = pd.DataFrame(rows)

    sns.barplot(data=df_plot, x='Feature', y='Value', hue='Model', palette=palette, ci=None, ax=ax)
    ax.set_ylabel('Disparate Impact' if metric == 'Disparate_Impact' else metric.replace('_',' ') + 'erence', fontsize=20)
    # larger tick labels for accessibility
    ax.tick_params(axis='x', labelsize=18, rotation=45)
    ax.tick_params(axis='y', labelsize=18)
    # subtle bar edges
    for p in ax.patches:
        p.set_edgecolor('#444444')
        p.set_linewidth(0.4)
    # add subplot label in the top-right corner
    ax.text(0.98, 0.95, label, transform=ax.transAxes, ha='right', va='top', fontsize=22, fontweight='bold')

# remove duplicate legends: place a single legend below the plots
handles, labels = axes[0].get_legend_handles_labels()
for ax in axes:
    ax.get_legend().remove()
leg = fig.legend(handles, preferred_order, title='Model', loc='lower center', ncol=len(models)/2, frameon=False, prop={'size':18})
if leg is not None:
    try:
        leg.get_title().set_fontsize(20)
    except Exception:
        pass

# make x-axis label larger on the shared x-axis (bottom subplot)
axes[-1].set_xlabel('Feature', fontsize=20)
# Set title for subplots
fig.suptitle('Fairness Metrics for Each Model and Feature',fontsize=22)
plt.tight_layout(rect=[0,0.07,1,0.99])
plt.show()

In [ ]:
import pprint
pprint.pprint(fairness_outputs)